# Task 2 — Data Modelling

This task focuses on analyzing the structure of a relational database using Microsoft SQL Server. The analysis includes checking the database design for the Third Normal Form (3NF), classifying the database as an OLTP or OLAP system, evaluating the structure of the Measurement table for analytical queries, and rewriting an aggregation query without using ROLLUP or CUBE.

## Task Description

1. Look attentively at logical diagram of the sandbox database. Consider that each field is atomic, i.e. all the data is at least in the 1st NF, could you tell if the 3rd NF is broken and where? How would you correct this?

2. Could you classify the sandbox DB model in terms of OLAP/OLTP system? Why do you think so?

3. Look carefully around Measurement table and related ones. What do you think, is the structure optimal (or close to optimal) to retrieve aggregated measurement information (like average temperature, etc.) by Cities and Countries? Could you show your reasoning?

4. Assume you need aggregate information provided by the query (its code is also available in the file ‘02 - rollup.sql’) that is based on auxiliary view (its code is in the file ’01 – auxiliary view.sql’):

SELECT
        city_name, [year], [month], [day],
        AVG (temperature) AS averageTemperature
FROM dbo.view_TemperatureMeasurement
GROUP BY ROLLUP (city_name, [year], [month], [day]);

Could you rewrite the query based on regular GROUP BY operation (it doesn’t matter how complex the result query will be) without using ROLLUP or CUBE?

## 1.1 Database Logical Model

The following logical model represents the database used
for the task.

![Database Logical Model](images/model.png)

## 1.2 Original Script — Auxiliary View

In [10]:
USE meteo_sandbox_db
GO

-- View is created just to simplify following examples
CREATE OR ALTER VIEW dbo.view_TemperatureMeasurement
AS
    SELECT
        m.geoPointId, gob.geoobjectId, tp.year, tp.month, tp.day, tp.hour, tp.minute,
        ROUND(m.temperature, 2) AS temperature,
        gob.national_name AS city_name, c.name AS country_name
    FROM dbo.Measurement AS m
        INNER JOIN dbo.TimePoint         AS tp   ON m.timePointId = tp.timePointId
        INNER JOIN dbo.GeoObjectGeoPoint AS gogp ON gogp.geoPointId = m.geoPointId
        INNER JOIN dbo.GeoObject         AS gob  ON gob.geoObjectId = gogp.geoObjectId
        INNER JOIN dbo.Country           AS c    ON c.countryId = gob.countryId

    -- filter condition is used here just to speed up the example execution
    WHERE tp.year = 2021 AND gob.geoObjectId IN (12018310, 11931712, 12033947, 12089437)     -- Polish cities with population > 600 K
GO

SELECT TOP 100 *
FROM dbo.view_TemperatureMeasurement
WHERE [hour] IN (0, 6, 12, 18) AND [minute] = 0     -- filter here just to minimize the result recordset


Commands completed successfully.

Commands completed successfully.

(100 rows affected)

geoPointId | geoobjectId | year | month | day | hour | minute | temperature | city_name | country_name
-----------+-------------+------+-------+-----+------+--------+-------------+-----------+-------------
13         | 11931712    | 2021 | 1     | 1   | 0    | 0      | -1.8        | Warsaw    | Poland      
13         | 11931712    | 2021 | 1     | 1   | 6    | 0      | -3.8        | Warsaw    | Poland      
13         | 11931712    | 2021 | 1     | 1   | 12   | 0      | -0.9        | Warsaw    | Poland      
13         | 11931712    | 2021 | 1     | 1   | 18   | 0      | -1.7        | Warsaw    | Poland      
13         | 11931712    | 2021 | 1     | 2   | 0    | 0      | -1.4        | Warsaw    | Poland      
13         | 11931712    | 2021 | 1     | 2   | 6    | 0      | -1.2        | Warsaw    | Poland      
13         | 11931712    | 2021 | 1     | 2   | 12   | 0      | 1.3         | Warsaw   

## 1.3 Original Script — Rollup

In [12]:
USE meteo_sandbox_db
GO

SELECT TOP 100 
    city_name, [year], [month], [day],
    AVG (temperature) AS averageTemperature
FROM dbo.view_TemperatureMeasurement
GROUP BY ROLLUP (city_name, [year], [month], [day])

Commands completed successfully.

(100 rows affected)

city_name | year | month | day  | averageTemperature   
----------+------+-------+------+----------------------
Kraków    | 2021 | 1     | 1    | 0.24479166666666663  
Kraków    | 2021 | 1     | 2    | 1.1531250000000004   
Kraków    | 2021 | 1     | 3    | 1.843749999999998    
Kraków    | 2021 | 1     | 4    | 2.29375              
Kraków    | 2021 | 1     | 5    | -0.14166666666666675 
Kraków    | 2021 | 1     | 6    | -1.508333333333333   
Kraków    | 2021 | 1     | 7    | -1.4604166666666665  
Kraków    | 2021 | 1     | 8    | -2.3895833333333347  
Kraków    | 2021 | 1     | 9    | -2.022916666666667   
Kraków    | 2021 | 1     | 10   | -3.818749999999999   
Kraków    | 2021 | 1     | 11   | -5.093750000000001   
Kraków    | 2021 | 1     | 12   | -3.4114583333333344  
Kraków    | 2021 | 1     | 13   | -0.6093750000000003  
Kraków    | 2021 | 1     | 14   | -2.6093749999999996  
Kraków    | 2021 | 1     | 15   | -7.157291666666

## 1.4 3rd Normal Form (3NF) Analysis

The database appears to have atomic fields, so it satisfies the first normal form (1NF).
Examining the schema, the Measurement table stores only foreign keys such as geoPointId and timePointId, along with the measured value (temperature). Information about cities and countries is not stored directly in this table but is retrieved through related tables (GeoObjectGeoPoint, GeoObject, and Country).

This design avoids transitive dependencies, because non-key attributes (such as city or country names) are not dependent on other non-key attributes within the same table. Instead, they are stored in separate tables and accessed via foreign keys. As a result, each non-key attribute depends only on the primary key of its table.

Therefore, the database satisfies the third normal form (3NF). No correction is required, as the schema already prevents redundancy and maintains proper normalization.

## 1.5 OLAP vs OLTP Classification

The database schema is closer to an OLTP (Online Transaction Processing) system because it is highly normalized and designed to minimize redundancy. Data is distributed across multiple related tables, and relationships are maintained through foreign keys.

Such a structure is well-suited for transactional operations, including frequent inserts (e.g., new measurements), updates, and deletes. However, it requires multiple joins to retrieve aggregated data.

OLAP systems, in contrast, typically use denormalized schemas such as star or snowflake models to optimize analytical queries and aggregation performance. Since this schema relies on normalization rather than denormalization, it is more appropriate for transactional workloads than for analytical processing.


## 1.6 Measurement Table and Aggregation Optimization

Looking at the Measurement table and its related tables, the structure is logically correct but not optimal for aggregated queries such as calculating average temperature by city or country. To perform such aggregations, the system must join several tables (Measurement, TimePoint, GeoObjectGeoPoint, GeoObject, and Country).

While this approach ensures data consistency and normalization, it increases the computational cost of aggregation queries, especially on large datasets. Multiple joins can significantly impact performance when grouping and calculating aggregates.

For analytical purposes, performance could be improved by introducing an OLAP-oriented design, such as a star schema, where Measurement acts as a fact table and dimensions (Time, City, Country) are stored in denormalized form. Additionally, indexed or materialized views could be used to precompute frequent aggregations.

Overall, the current structure is suitable for OLTP operations but not optimized for efficient large-scale analytical queries.


## 1.7 GROUP BY Solution

In [14]:
USE meteo_sandbox_db;
GO

SELECT TOP 100
    city_name,
    [year],
    [month],
    [day],
    averageTemperature
FROM
(
    SELECT
        city_name,
        [year],
        [month],
        [day],
        AVG(temperature) AS averageTemperature
    FROM dbo.view_TemperatureMeasurement
    GROUP BY
        city_name,
        [year],
        [month],
        [day]

    UNION ALL

    SELECT
        city_name,
        [year],
        [month],
        NULL AS [day],
        AVG(temperature) AS averageTemperature
    FROM dbo.view_TemperatureMeasurement
    GROUP BY
        city_name,
        [year],
        [month]

    UNION ALL

    SELECT
        city_name,
        [year],
        NULL AS [month],
        NULL AS [day],
        AVG(temperature) AS averageTemperature
    FROM dbo.view_TemperatureMeasurement
    GROUP BY
        city_name,
        [year]

    UNION ALL

    SELECT
        city_name,
        NULL AS [year],
        NULL AS [month],
        NULL AS [day],
        AVG(temperature) AS averageTemperature
    FROM dbo.view_TemperatureMeasurement
    GROUP BY
        city_name

    UNION ALL

    SELECT
        NULL AS city_name,
        NULL AS [year],
        NULL AS [month],
        NULL AS [day],
        AVG(temperature) AS averageTemperature
    FROM dbo.view_TemperatureMeasurement
) AS Result;

Commands completed successfully.

(100 rows affected)

city_name | year | month | day | averageTemperature  
----------+------+-------+-----+---------------------
Wrocław   | 2021 | 3     | 4   | 3.9520833333333343  
Wrocław   | 2021 | 8     | 21  | 19.171875000000004  
Kraków    | 2021 | 2     | 18  | -0.2822916666666668 
Kraków    | 2021 | 3     | 10  | -2.2260416666666663 
Kraków    | 2021 | 4     | 9   | 4.894791666666669   
Łódź      | 2021 | 4     | 8   | 1.8572916666666668  
Łódź      | 2021 | 9     | 9   | 17.378124999999994  
Łódź      | 2021 | 10    | 6   | 12.697916666666664  
Łódź      | 2021 | 10    | 24  | 4.332291666666667   
Łódź      | 2021 | 12    | 31  | 6.454166666666662   
Wrocław   | 2021 | 8     | 17  | 16.29479166666667   
Wrocław   | 2021 | 10    | 8   | 7.661458333333333   
Kraków    | 2021 | 3     | 25  | 4.706249999999999   
Kraków    | 2021 | 6     | 5   | 17.497916666666665  
Kraków    | 2021 | 10    | 5   | 15.805208333333335  
Kraków    | 2021 | 10    | 